# One-dimensional interpolation with CUDA

This notebook compares nearest-neighbor and linear interpolation. Each CUDA thread evaluates one requested coordinate. Coordinates outside the sample interval use constant extension (the nearest endpoint value).

Run it in a Jupyter environment with an NVIDIA GPU, CUDA toolkit, and `nvcc`. The CUDA program is written to a `.cu` file, compiled, and run in the following cells.

In [1]:
!nvidia-smi
!nvcc --version

Sun Nov 15 18:27:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA program

The host program checks both GPU results against the corresponding CPU formulas. The small example includes interior points, a midpoint, and coordinates outside the sample range so that the boundary rule is visible.

In [2]:
%%writefile interpolation_1d.cu
#include <cuda_runtime.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>

#define CUDA_CHECK(call) do { \
    cudaError_t error = (call); \
    if (error != cudaSuccess) { \
        fprintf(stderr, "CUDA error %s at %s:%d\n", \
                cudaGetErrorString(error), __FILE__, __LINE__); \
        exit(EXIT_FAILURE); \
    } \
} while (0)

__device__ float clamp_coordinate(float x, int n)
{
    return fminf(fmaxf(x, 0.0f), float(n - 1));
}

__device__ float nearest_value(const float *samples, int n, float x)
{
    x = clamp_coordinate(x, n);
    const int i = int(floorf(x + 0.5f));
    return samples[i];
}

__device__ float linear_value(const float *samples, int n, float x)
{
    x = clamp_coordinate(x, n);
    const int i0 = int(floorf(x));
    const int i1 = (i0 + 1 < n) ? i0 + 1 : n - 1;
    const float t = x - float(i0);
    return (1.0f - t) * samples[i0] + t * samples[i1];
}

__global__ void interpolate_1d(const float *samples, const float *x,
                               float *nearest, float *linear,
                               int nSamples, int nQueries)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < nQueries) {
        nearest[i] = nearest_value(samples, nSamples, x[i]);
        linear[i] = linear_value(samples, nSamples, x[i]);
    }
}

float host_nearest(const float *samples, int n, float x)
{
    x = fminf(fmaxf(x, 0.0f), float(n - 1));
    return samples[int(floorf(x + 0.5f))];
}

float host_linear(const float *samples, int n, float x)
{
    x = fminf(fmaxf(x, 0.0f), float(n - 1));
    const int i0 = int(floorf(x));
    const int i1 = (i0 + 1 < n) ? i0 + 1 : n - 1;
    const float t = x - float(i0);
    return (1.0f - t) * samples[i0] + t * samples[i1];
}

int main()
{
    const int nSamples = 5;
    const int nQueries = 7;
    const int threads = 128;
    const float h_samples[nSamples] = {0.0f, 0.2f, 0.4f, 0.6f, 0.8f};
    const float h_x[nQueries] = {-0.6f, -0.1f, 0.6f, 1.5f,
                                  2.1f, 2.9f, 4.7f};
    float h_nearest[nQueries] = {};
    float h_linear[nQueries] = {};

    float *d_samples = nullptr, *d_x = nullptr;
    float *d_nearest = nullptr, *d_linear = nullptr;
    CUDA_CHECK(cudaMalloc(&d_samples, nSamples * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_x, nQueries * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_nearest, nQueries * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_linear, nQueries * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_samples, h_samples, nSamples * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, nQueries * sizeof(float),
                          cudaMemcpyHostToDevice));

    const int blocks = (nQueries + threads - 1) / threads;
    interpolate_1d<<<blocks, threads>>>(d_samples, d_x, d_nearest, d_linear,
                                        nSamples, nQueries);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(h_nearest, d_nearest, nQueries * sizeof(float),
                          cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_linear, d_linear, nQueries * sizeof(float),
                          cudaMemcpyDeviceToHost));

    bool pass = true;
    printf("index    x       nearest    linear\n");
    for (int i = 0; i < nQueries; ++i) {
        const float refNearest = host_nearest(h_samples, nSamples, h_x[i]);
        const float refLinear = host_linear(h_samples, nSamples, h_x[i]);
        pass = pass && fabsf(h_nearest[i] - refNearest) < 1.0e-6f;
        pass = pass && fabsf(h_linear[i] - refLinear) < 1.0e-6f;
        printf("%5d  %6.2f   %7.3f    %7.3f\n",
               i, h_x[i], h_nearest[i], h_linear[i]);
    }
    printf("validation = %s\n", pass ? "PASS" : "FAIL");

    CUDA_CHECK(cudaFree(d_samples));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_nearest));
    CUDA_CHECK(cudaFree(d_linear));
    return pass ? EXIT_SUCCESS : EXIT_FAILURE;
}


Writing interpolation_1d.cu


In [3]:
!nvcc -O2 -std=c++14 -arch=native interpolation_1d.cu -o interpolation_1d
!./interpolation_1d

index    x       nearest    linear
    0   -0.60     0.000      0.000
    1   -0.10     0.000      0.000
    2    0.60     0.200      0.120
    3    1.50     0.400      0.300
    4    2.10     0.400      0.420
    5    2.90     0.600      0.580
    6    4.70     0.800      0.800
validation = PASS


Each query is independent, so it can be assigned to a separate thread. The grid is rounded up and the kernel checks `i < nQueries`; this also works when the number of queries is not a multiple of the block size.